# Team Age Distribution Analysis

Analyze the age distribution of players across all 10 teams in the Montenegrin First League.

**Note:** Player ages are calculated at the time of each match, not current age. This provides a more accurate historical analysis weighted by minutes played in each game.

In [22]:
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [23]:
# Path configuration
DATA_DIR = Path("..") / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUT_DIR = Path("..") / "outputs" / "figures"

OUT_DIR = RAW_DATA_DIR / "mt1cfl_2526"
RAW_DIR = OUT_DIR / "raw_by_match"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [24]:
# Load matches to get team mapping, timestamps, and filter postponed matches
matches_df = pd.read_csv(OUT_DIR / "matches.csv")
matches_df_clean = matches_df[matches_df['status.description'] != 'Postponed'].copy()
valid_match_ids = set(matches_df_clean['id'].astype(str))

# Create team mapping
team_mapping = {}
for _, row in matches_df_clean.iterrows():
    team_mapping[row['homeTeam.id']] = row['homeTeam.name']
    team_mapping[row['awayTeam.id']] = row['awayTeam.name']

# Create match timestamp mapping
match_timestamps = dict(zip(matches_df_clean['id'].astype(str), matches_df_clean['startTimestamp']))

print(f"✅ Found {len(team_mapping)} teams in the league")
print(f"✅ Loaded timestamps for {len(match_timestamps)} matches")
print(f"\nTeams: {', '.join(sorted(team_mapping.values()))}")

✅ Found 10 teams in the league
✅ Loaded timestamps for 95 matches

Teams: FK Arsenal Tivat, FK Bokelj Kotor, FK Budućnost Podgorica, FK Dečić Tuzi, FK Jedinstvo Bijelo Polje, FK Jezero, FK Mornar Bar, FK Sutjeska Nikšić, Mladost DG, OFK Petrovac


In [25]:
# Extract player information including birth dates from lineups
player_info_list = []

for match_dir in RAW_DIR.glob("*"):
    # Skip postponed matches
    if match_dir.name not in valid_match_ids:
        continue
    
    lineups_file = match_dir / "lineups.json"
    if lineups_file.exists():
        try:
            with open(lineups_file, encoding='utf-8') as f:
                lineups_data = json.load(f)
            
            # Get match info for team IDs and names from match metadata
            match_info = matches_df_clean[matches_df_clean['id'] == int(match_dir.name)].iloc[0]
            team_ids = {
                'home': int(match_info['homeTeam.id']),
                'away': int(match_info['awayTeam.id'])
            }
            team_names = {
                'home': match_info['homeTeam.name'],
                'away': match_info['awayTeam.name']
            }
            
            # Extract from both teams
            for team_side in ['home', 'away']:
                if team_side in lineups_data:
                    team_data = lineups_data[team_side]
                    players = team_data.get('players', [])
                    
                    for player_entry in players:
                        player = player_entry.get('player', {})
                        stats = player_entry.get('statistics', {})
                        
                        # Only include players who actually played
                        minutes_played = stats.get('minutesPlayed', 0)
                        if minutes_played > 0:
                            player_info = {
                                'match_id': match_dir.name,
                                'player_id': player.get('id'),
                                'player_name': player.get('name'),
                                'team_id': team_ids[team_side],
                                'team_name': team_names[team_side],
                                'dateOfBirthTimestamp': player.get('dateOfBirthTimestamp'),
                                'minutes_played': minutes_played
                            }
                            player_info_list.append(player_info)
        except Exception as e:
            print(f"Error processing {lineups_file}: {e}")

players_df = pd.DataFrame(player_info_list)
print(f"\n✅ Loaded {len(players_df)} player-match records")
print(f"📊 Unique players: {players_df['player_id'].nunique()}")

# Check for missing birth dates
missing_dob = players_df['dateOfBirthTimestamp'].isna().sum()
print(f"\n⚠️  Players without birth date: {missing_dob} ({missing_dob/len(players_df)*100:.1f}%)")

# Show sample
display(players_df.head(10))


✅ Loaded 2933 player-match records
📊 Unique players: 253

⚠️  Players without birth date: 45 (1.5%)


,match_id,player_id,player_name,team_id,team_name,dateOfBirthTimestamp,minutes_played
0,14147297,927881,Stefan Spasojević,7581,FK Bokelj Kotor,7.460640e+08,90
1,14147297,314142,Stefan Vico,7581,FK Bokelj Kotor,7.939296e+08,46
2,14147297,94246,Igor Ćuković,7581,FK Bokelj Kotor,7.393248e+08,59
3,14147297,57463,Saša Balić,7581,FK Bokelj Kotor,6.335712e+08,90
4,14147297,909420,Marko Čavor,7581,FK Bokelj Kotor,9.311328e+08,90
5,14147297,1381377,Kristijan Radunović,7581,FK Bokelj Kotor,1.094602e+09,90
6,14147297,1486746,Alhassan Baba Musah,7581,FK Bokelj Kotor,9.524736e+08,90
7,14147297,807177,Žarko Vilotijević,7581,FK Bokelj Kotor,8.207136e+08,46
8,14147297,572018,Velizar Janketić,7581,FK Bokelj Kotor,8.480160e+08,90
9,14147297,813112,Luka Maraš,7581,FK Bokelj Kotor,8.328960e+08,71


In [26]:
# Calculate ages at the time of each match
players_df_clean = players_df.dropna(subset=['dateOfBirthTimestamp']).copy()

# Add match timestamp
players_df_clean['match_timestamp'] = players_df_clean['match_id'].map(match_timestamps)

# Convert timestamps to datetime
players_df_clean['birth_date'] = pd.to_datetime(players_df_clean['dateOfBirthTimestamp'], unit='s')
players_df_clean['match_date'] = pd.to_datetime(players_df_clean['match_timestamp'], unit='s')

# Calculate age at time of match
players_df_clean['age_at_match'] = ((players_df_clean['match_date'] - players_df_clean['birth_date']).dt.days / 365.25).astype(int)

print(f"✅ Calculated ages for {len(players_df_clean)} player-match records")
print(f"\nSample of age calculations:")
display(players_df_clean[['player_name', 'team_name', 'match_date', 'birth_date', 'age_at_match', 'minutes_played']].head(10))

# Aggregate by player - weighted average age
player_ages = players_df_clean.groupby('player_id').agg({
    'player_name': 'first',
    'team_id': 'last',  # Most recent team
    'team_name': 'last',
    'age_at_match': lambda x: (x * players_df_clean.loc[x.index, 'minutes_played']).sum() / players_df_clean.loc[x.index, 'minutes_played'].sum(),  # Weighted average age
    'minutes_played': 'sum'
}).reset_index()

player_ages['age_at_match'] = player_ages['age_at_match'].round(1)

# Create age categories
def categorize_age(age):
    if age < 21:
        return 'Under 21'
    elif age <= 24:
        return '21 to 24'
    elif age <= 28:
        return '25 to 28'
    else:
        return '29+'

player_ages['age_category'] = player_ages['age_at_match'].apply(categorize_age)

print(f"\n✅ Processed {len(player_ages)} unique players with age data")
print(f"\nAge distribution (weighted by minutes):")
print(player_ages['age_category'].value_counts().sort_index())

print(f"\nAge statistics (weighted average age at match):")
print(f"  Youngest: {player_ages['age_at_match'].min():.1f} years")
print(f"  Oldest: {player_ages['age_at_match'].max():.1f} years")
print(f"  Average: {player_ages['age_at_match'].mean():.1f} years")
print(f"  Median: {player_ages['age_at_match'].median():.1f} years")

print(f"\nSample of players with weighted ages:")
display(player_ages.head(10))

✅ Calculated ages for 2888 player-match records

Sample of age calculations:


,player_name,team_name,match_date,birth_date,age_at_match,minutes_played
0,Stefan Spasojević,FK Bokelj Kotor,2025-08-10 18:00:00,1993-08-23,31,90
1,Stefan Vico,FK Bokelj Kotor,2025-08-10 18:00:00,1995-02-28,30,46
2,Igor Ćuković,FK Bokelj Kotor,2025-08-10 18:00:00,1993-06-06,32,59
3,Saša Balić,FK Bokelj Kotor,2025-08-10 18:00:00,1990-01-29,35,90
4,Marko Čavor,FK Bokelj Kotor,2025-08-10 18:00:00,1999-07-05,26,90
5,Kristijan Radunović,FK Bokelj Kotor,2025-08-10 18:00:00,2004-09-08,20,90
6,Alhassan Baba Musah,FK Bokelj Kotor,2025-08-10 18:00:00,2000-03-08,25,90
7,Žarko Vilotijević,FK Bokelj Kotor,2025-08-10 18:00:00,1996-01-04,29,46
8,Velizar Janketić,FK Bokelj Kotor,2025-08-10 18:00:00,1996-11-15,28,90
9,Luka Maraš,FK Bokelj Kotor,2025-08-10 18:00:00,1996-05-24,29,71



✅ Processed 244 unique players with age data

Age distribution (weighted by minutes):
age_category
21 to 24    67
25 to 28    52
29+         78
Under 21    47
Name: count, dtype: int64

Age statistics (weighted average age at match):
  Youngest: 16.0 years
  Oldest: 38.0 years
  Average: 25.6 years
  Median: 25.0 years

Sample of players with weighted ages:


,player_id,player_name,team_id,team_name,age_at_match,minutes_played,age_category
0,44305,Draško Božović,6226,FK Dečić Tuzi,37.0,701,29+
1,54752,Žarko Korać,6219,FK Jedinstvo Bijelo Polje,38.0,1384,29+
2,55534,Aleksandar Kapisoda,6216,OFK Petrovac,35.0,336,29+
3,55655,Milan Mijatović,5143,FK Budućnost Podgorica,38.0,990,29+
4,55668,Miloš Dragojević,6226,FK Dečić Tuzi,36.0,942,29+
5,57339,Dejan Boljević,6216,OFK Petrovac,35.0,1573,29+
6,57341,Petar Grbić,5143,FK Budućnost Podgorica,36.9,622,29+
7,57363,Vladan Giljen,6224,FK Sutjeska Nikšić,35.1,1620,29+
8,57463,Saša Balić,7581,FK Bokelj Kotor,35.0,337,29+
9,57729,Luka Mirković,5143,FK Budućnost Podgorica,34.0,4,29+


In [27]:
# Top 20 players by total minutes played
print("=== TOP 20 PLAYERS BY MINUTES PLAYED ===\n")

top_20_players = player_ages.sort_values('minutes_played', ascending=False).head(20).copy()
top_20_players['rank'] = range(1, 21)

# Select and reorder columns for display
top_20_display = top_20_players[['rank', 'player_name', 'team_name', 'minutes_played', 'age_at_match', 'age_category']].copy()

print(f"Showing players with most minutes in the 2025/26 season")
print(f"Ages shown are weighted averages at time of matches played\n")
display(top_20_display)

# Quick stats on top 20
print(f"\n📊 Top 20 Players Statistics:")
print(f"  Average age: {top_20_players['age_at_match'].mean():.1f} years")
print(f"  Youngest: {top_20_players[top_20_players['age_at_match'] == top_20_players['age_at_match'].min()].iloc[0]['player_name']} ({top_20_players['age_at_match'].min():.1f} years)")
print(f"  Oldest: {top_20_players[top_20_players['age_at_match'] == top_20_players['age_at_match'].max()].iloc[0]['player_name']} ({top_20_players['age_at_match'].max():.1f} years)")
print(f"  Total minutes: {top_20_players['minutes_played'].sum():,} ({top_20_players['minutes_played'].sum()/(95*90):.1%} of all possible minutes)")

print(f"\n🎯 Age Category Breakdown (Top 20):")
print(top_20_players['age_category'].value_counts().sort_index())

=== TOP 20 PLAYERS BY MINUTES PLAYED ===

Showing players with most minutes in the 2025/26 season
Ages shown are weighted averages at time of matches played



,rank,player_name,team_name,minutes_played,age_at_match,age_category
170,1,Nemanja Jovičić,FK Jezero,1710,25.0,25 to 28
99,2,Jonathan Dresaj,FK Dečić Tuzi,1708,25.0,25 to 28
120,3,Aleksa Ćetković,FK Arsenal Tivat,1708,21.0,21 to 24
18,4,Milivoje Raičević,FK Jezero,1659,32.0,29+
51,5,Bojan Zogović,FK Arsenal Tivat,1620,36.0,29+
27,6,Marko Kordić,OFK Petrovac,1620,30.0,29+
14,7,Andrija Kaluđerović,FK Mornar Bar,1620,31.3,29+
7,8,Vladan Giljen,FK Sutjeska Nikšić,1620,35.1,29+
36,9,Stefan Vico,FK Bokelj Kotor,1608,30.0,29+
171,10,Arsenije Čepić,FK Jedinstvo Bijelo Polje,1601,21.2,21 to 24



📊 Top 20 Players Statistics:
  Average age: 27.3 years
  Youngest: Medo Juković (20.0 years)
  Oldest: Bojan Zogović (36.0 years)
  Total minutes: 31,848 (372.5% of all possible minutes)

🎯 Age Category Breakdown (Top 20):
age_category
21 to 24    6
25 to 28    4
29+         9
Under 21    1
Name: count, dtype: int64


In [28]:
# For team distribution, we need to use the match-level data with age categories
players_df_clean['age_category'] = players_df_clean['age_at_match'].apply(categorize_age)

# Calculate team age distributions (weighted by minutes played)
team_age_dist = players_df_clean.groupby(['team_name', 'age_category']).agg({
    'minutes_played': 'sum',
    'player_id': 'nunique'
}).reset_index()

team_age_dist.columns = ['team_name', 'age_category', 'total_minutes', 'player_count']

# Calculate percentage of total minutes per team
team_totals = team_age_dist.groupby('team_name')['total_minutes'].sum().reset_index()
team_totals.columns = ['team_name', 'team_total_minutes']

team_age_dist = team_age_dist.merge(team_totals, on='team_name')
team_age_dist['percentage'] = (team_age_dist['total_minutes'] / team_age_dist['team_total_minutes'] * 100).round(0).astype(int)

# Pivot for visualization
age_pivot = team_age_dist.pivot(index='team_name', columns='age_category', values='percentage').fillna(0)

# Ensure all age categories exist
for cat in ['Under 21', '21 to 24', '25 to 28', '29+']:
    if cat not in age_pivot.columns:
        age_pivot[cat] = 0

# Reorder columns
age_pivot = age_pivot[['Under 21', '21 to 24', '25 to 28', '29+']]

# Drop any aggregate or unknown teams from the table/plot
if 'Unknown' in age_pivot.index:
    age_pivot = age_pivot.drop(index='Unknown')

# Calculate team average ages
team_avg_age = players_df_clean.groupby('team_name').apply(
    lambda x: (x['age_at_match'] * x['minutes_played']).sum() / x['minutes_played'].sum()
).reset_index()
team_avg_age.columns = ['team_name', 'avg_age']
team_avg_age['avg_age'] = team_avg_age['avg_age'].round(1)

# Also drop 'Unknown' from team average ages, if present
team_avg_age = team_avg_age[team_avg_age['team_name'] != 'Unknown']

# Load league standings to order teams by position
standings_df = pd.read_csv(PROCESSED_DATA_DIR / 'league_standings.csv')
team_position_map = dict(zip(standings_df['team_name'], standings_df['position']))
print("✅ Loaded league standings for team ordering")

# Sort teams so that 1st place is at the top bar and last place at the bottom
# reverse=True puts position 10 first (plots at bottom), position 1 last (plots at top)
team_order = sorted(age_pivot.index.tolist(), key=lambda x: team_position_map.get(x, 99), reverse=True)
age_pivot = age_pivot.reindex(team_order)

# Add position numbers to team names for display
age_pivot_display = age_pivot.copy()
age_pivot_display.index = [f"{team_position_map.get(team, '?')}. {team}" for team in age_pivot_display.index]

print("\n📊 Team Age Distribution (weighted by minutes played):")
print("Teams ordered by current league position (1st to 10th)\n")
display(age_pivot_display)

print("\n📈 Team Average Ages (at time of matches):")
team_avg_age = team_avg_age.sort_values('avg_age')
display(team_avg_age)

✅ Loaded league standings for team ordering

📊 Team Age Distribution (weighted by minutes played):
Teams ordered by current league position (1st to 10th)



C:\Users\pavle\AppData\Local\Temp\ipykernel_25840\3373391784.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  team_avg_age = players_df_clean.groupby('team_name').apply(


age_category,Under 21,21 to 24,25 to 28,29+
10. FK Jedinstvo Bijelo Polje,18,33,30,19
9. FK Bokelj Kotor,16,19,28,37
8. FK Arsenal Tivat,10,19,13,59
7. FK Budućnost Podgorica,16,22,16,46
6. Mladost DG,22,28,27,24
5. OFK Petrovac,16,24,36,24
4. FK Jezero,13,26,25,36
3. FK Dečić Tuzi,10,24,35,30
2. FK Mornar Bar,18,10,27,45
1. FK Sutjeska Nikšić,14,44,6,36



📈 Team Average Ages (at time of matches):


,team_name,avg_age
8,Mladost DG,24.7
4,FK Jedinstvo Bijelo Polje,25.0
9,OFK Petrovac,26.0
1,FK Bokelj Kotor,26.1
5,FK Jezero,26.1
7,FK Sutjeska Nikšić,26.1
3,FK Dečić Tuzi,26.6
2,FK Budućnost Podgorica,27.0
6,FK Mornar Bar,27.4
0,FK Arsenal Tivat,28.1
